In [ ]:
import os
import tensorflow as tf

os.environ["KERAS_BACKEND"] = "torch"

import torch
import keras
import numpy as np

keras.utils.set_random_seed(102953)

In [ ]:
keras.backend.backend()

In [ ]:
%load_ext tensorboard
# now available at http://localhost:6006/?

In [ ]:
import json

with open('./config/keras_nn.json') as keras_nn_config:
    CONFIG = json.load(keras_nn_config)
    print("config loaded")

In [ ]:
from data_loader import get_ml_cup_data
from sklearn.preprocessing import MinMaxScaler

train_loader, test_loader, input_size, output_size = get_ml_cup_data(CONFIG["batchSize"], scaler=MinMaxScaler())

In [ ]:
from keras import Sequential
from keras.layers import Input, Dense

In [ ]:
model = Sequential([
    Input(shape=(input_size,)),
    # Dense layers are fully connected layers
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(output_size)
])

model.compile(loss='mean_squared_error', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
from keras.callbacks import EarlyStopping, TensorBoard
import datetime

def log_dir(name, append:str=None):
    BASE = f"logs/{name}/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    if append:
        BASE += "_" + append
    return BASE

In [ ]:
tensorboard_cb = TensorBoard(
    log_dir=log_dir("fit", "ES_OPTIMIZER_OFF"),
    histogram_freq=1,
    write_graph=True,
    write_images=False,
)

early_stopping_cb = EarlyStopping(
    monitor="accuracy",
    patience=10,
    restore_best_weights=True,
)

In [ ]:
history = model.fit(train_loader, epochs=100, callbacks=[tensorboard_cb, early_stopping_cb])

In [ ]:
%tensorboard --logdir logs/fit

In [ ]:
# evaluate model
results = model.evaluate(test_loader, return_dict=True)
print(results)

In [ ]:
# write logs to a separate folder
writer = tf.summary.create_file_writer(log_dir("eval", "ES_OPTIMIZER_OFF"))

with writer.as_default():
    for k, v in results.items():
        tf.summary.scalar(k, v, step=0)

writer.close()

In [ ]:
%tensorboard --logdir logs/eval